# Transfer Learning with Pretrained ResNet18

This notebook demonstrates transfer learning for the six-class Intel Image Classification dataset using a pretrained ResNet18. The existing project data loaders and training utilities are used, and only one training epoch is run as an executable pipeline demonstration.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    candidate for candidate in project_candidates
    if (candidate / "src" / "image_classifier").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from image_classifier.config import BATCH_SIZE, DEVICE, LEARNING_RATE
from image_classifier.data_loader import create_dataloaders
from image_classifier.models.transfer_learning import create_resnet18
from image_classifier.training.train import train_one_epoch, validate_one_epoch

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Source directory added to sys.path: {SRC_DIR.resolve()}")

Project root: C:\Users\ArshadShaik\Desktop\image-classification-cnn
Source directory added to sys.path: C:\Users\ArshadShaik\Desktop\image-classification-cnn\src


## 1. Connect to the project source

In [2]:
train_loader, val_loader, test_loader = create_dataloaders()

if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)

print(f"Training dataset size: {len(train_loader.dataset):,}")
print(f"Validation dataset size: {len(val_loader.dataset):,}")
print(f"Test dataset size: {len(test_loader.dataset):,}")
print(f"Batch size: {train_loader.batch_size} (configured value: {BATCH_SIZE})")
print(f"Device: {device}")

Training dataset size: 11,227
Validation dataset size: 2,807
Test dataset size: 3,000
Batch size: 64 (configured value: 64)
Device: cpu


## Create Pretrained ResNet18

The pretrained ResNet18 backbone is used as the starting point. The final classification layer is replaced with a layer containing six outputs for the six Intel Image Classification classes.

In [3]:
model = create_resnet18(
    num_classes=6,
    freeze_backbone=True,
)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ArshadShaik/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## Classification Head

In [5]:
print("Classification head:")
print(model.fc)

Classification head:
Linear(in_features=512, out_features=6, bias=True)


## Trainable Parameters

In [4]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

backbone_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if not name.startswith("fc.")
]
classifier_parameters = list(model.fc.parameters())
backbone_is_frozen = all(not parameter.requires_grad for parameter in backbone_parameters)
classifier_is_trainable = all(parameter.requires_grad for parameter in classifier_parameters)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Pretrained backbone frozen: {backbone_is_frozen}")
print(f"Final classifier trainable: {classifier_is_trainable}")
print(f"Frozen backbone tensors: {len(backbone_parameters):,}")
print(f"Trainable classifier tensors: {len(classifier_parameters):,}")

assert backbone_is_frozen
assert classifier_is_trainable

Total parameters: 11,179,590
Trainable parameters: 3,078
Pretrained backbone frozen: True
Final classifier trainable: True
Frozen backbone tensors: 60
Trainable classifier tensors: 2


## Forward Pass

In [6]:
images, labels = next(iter(train_loader))
model = model.to(device)
images = images.to(device)
labels = labels.to(device)
loss_fn = nn.CrossEntropyLoss()

with torch.no_grad():
    outputs = model(images)
    batch_loss = loss_fn(outputs, labels)

print(f"Input tensor shape: {tuple(images.shape)}")
print(f"Output tensor shape: {tuple(outputs.shape)}")
print(f"CrossEntropyLoss for the batch: {batch_loss.item():.4f}")

Input tensor shape: (64, 3, 224, 224)
Output tensor shape: (64, 6)
CrossEntropyLoss for the batch: 1.9227


## Interpretation

Transfer learning starts with a model that learned visual features from a large, general image dataset and adapts it to a new task. Pretrained ResNet18 is useful here because its convolutional backbone already provides strong low-level and mid-level image features, reducing the amount of task-specific training needed. Freezing the backbone means its pretrained weights are not updated during backpropagation. In this demonstration, only the replacement six-class classifier is trained, which keeps the experiment short and isolates the transfer-learning workflow.

## Training configuration

Only parameters with `requires_grad=True` are passed to the optimizer, so this demonstration updates the replacement classifier while keeping the pretrained backbone frozen.

In [7]:
trainable_parameters_for_optimizer = [
    parameter for parameter in model.parameters() if parameter.requires_grad
]
optimizer = torch.optim.Adam(trainable_parameters_for_optimizer, lr=LEARNING_RATE)

print(f"Model: {model.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Device: {device}")
print(f"Frozen backbone: {backbone_is_frozen}")
print(f"Optimizer parameter tensors: {len(trainable_parameters_for_optimizer)}")

Model: ResNet
Optimizer: Adam
Learning rate: 0.001
Device: cpu
Frozen backbone: True
Optimizer parameter tensors: 2


## One training epoch and validation

The existing project training functions run one complete pass over the training split followed by validation.

In [8]:
train_loss, train_accuracy = train_one_epoch(
    model=model,
    dataloader=train_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
)

validation_loss, validation_accuracy = validate_one_epoch(
    model=model,
    dataloader=val_loader,
    loss_fn=loss_fn,
    device=device,
)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {validation_accuracy:.2%}")

Batch 25/176
Batch 50/176
Batch 75/176
Batch 100/176
Batch 125/176
Batch 150/176
Batch 175/176
Batch 176/176
Training loss: 0.6854
Training accuracy: 78.74%
Validation loss: 0.3639
Validation accuracy: 89.13%


## Conceptual comparison

| Aspect | Custom CNN | ResNet18 transfer learning |
|---|---|---|
| Architecture | Compact project-specific convolutional network | Deep residual network with skip connections |
| Initialization | Randomly initialized weights | ImageNet-pretrained weights plus a new six-class head |
| Feature extraction | Learns visual features from the Intel dataset | Reuses pretrained visual features from the frozen backbone |
| Trainable parameters | All model parameters are trainable | Only the final classifier is trainable in this notebook |
| Training approach | Train the full network from scratch | Train the replacement classifier for one demonstration epoch |